# Graddy (2006) — Fulton Fish Market demand elasticity via IV

**Paper:** Graddy, K. (2006). *Markets: The Fulton Fish Market.* J. Economic Perspectives 20(2), 207–220.

**Design:** IV / 2SLS. **Data:** a *simulated* DGP (the original data is on Graddy's website; StatsPAI ships a deterministic replica with a known true price elasticity of **−0.95**). This notebook is a known-truth IV recovery demo, not a real-data replication.

**What we reproduce:** weather (wave height) instruments price to recover the demand elasticity despite supply/demand simultaneity.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')  # headless-safe (notebooks run under nbclient in CI)
import matplotlib.pyplot as plt
import numpy as np
import statspai as sp
print('statspai', sp.__version__)

In [ ]:
df, _ = sp.replicate('graddy_2006')
true_elasticity = df.attrs.get('true_elasticity')
print('simulated DGP; true elasticity =', true_elasticity)
df.head()

In [ ]:
ols = sp.regress('log_quantity ~ log_price + mon + tue + wed + thu',
                 data=df, robust='hc1')
iv = sp.ivreg('log_quantity ~ mon + tue + wed + thu + '
              '(log_price ~ wave_height)', data=df, robust='hc1')
ols_e = float(ols.params['log_price'])
iv_e = float(iv.params['log_price'])
print(f'OLS elasticity     : {ols_e:.3f}')
print(f'IV (wave) elasticity: {iv_e:.3f}  (true {true_elasticity})')

In [ ]:
import pandas as pd
tab = pd.DataFrame([
    ['OLS elasticity', ols_e, 'biased by simultaneity'],
    ['IV (wave) elasticity', iv_e, f'recovers true {true_elasticity}'],
], columns=['quantity', 'StatsPAI', 'note'])
tab

In [ ]:
# --- DRIFT GUARD ---
# Deterministic simulated DGP (seed 42): IV brackets the true -0.95.
assert -1.3 < iv_e < -0.7, iv_e
assert true_elasticity == -0.95
print(f'OK: IV recovers the true elasticity ({iv_e:.3f} ~ -0.95).')

**Result.** On the deterministic replica, the wave-height IV recovers a price elasticity of demand bracketing the true −0.95, illustrating identification under supply/demand simultaneity. Because the data is simulated, this is a teaching/known-truth demo rather than a parity claim against Graddy's original numbers.